In [0]:
from pyspark.sql import functions as F

# -------------------------
# 1️⃣ Load masked Gold/Silver tables
# -------------------------
users = spark.table("hetul_catalog_ott_02.gold.masked_gold_user_dim")
events = spark.table("hetul_catalog_ott_02.silver.masked_silver_events")
ads = spark.table("hetul_catalog_ott_02.gold.gold_ad_attribution_fact")

# -------------------------
# 2️⃣ Engagement Features
# -------------------------
engagement = events.groupBy("user_id").agg(
    F.count("event_id").alias("session_count"),
    F.sum("watch_span").alias("total_watch_time_sec"),
    (F.sum("watch_span") / F.count("event_id")).alias("avg_watch_time_sec"),
    F.countDistinct("device").alias("distinct_devices"),
    F.countDistinct("content_id").alias("distinct_content_watched"),
    F.countDistinct("errors").alias("distinct_error_types")
)

# -------------------------
# 3️⃣ Error / Quality Features
# -------------------------
error_features = events.groupBy("user_id").pivot("errors").count().na.fill(0)
# Example columns: crash, buffering, network_drop, no_error

# Avg bitrate per user
bitrate_avg = events.groupBy("user_id").agg(F.avg("bitrate").alias("avg_bitrate"))

# -------------------------
# 4️⃣ Ad Engagement Features
# -------------------------
ad_features = ads.groupBy("user_id").agg(
    F.sum("impressions").alias("ad_impressions"),
    F.sum("clicks").alias("ad_clicks"),
    F.sum("revenue").alias("ad_revenue"),
    (F.sum("clicks") / F.sum("impressions")).alias("ad_ctr")
)

# -------------------------
# 5️⃣ Combine All Features
# -------------------------
ml_features = users.join(engagement, "user_id", "left") \
                   .join(error_features, "user_id", "left") \
                   .join(bitrate_avg, "user_id", "left") \
                   .join(ad_features, "user_id", "left") \
                   .fillna(0)  # Fill nulls for users with no events/ads

# -------------------------
# 6️⃣ Add Target Variable
# -------------------------
ml_features = ml_features.withColumnRenamed("churn_flag", "label")  # 0/1

# -------------------------
# 7️⃣ Null-Safe Manual Categorical Encoding
# -------------------------
categorical_cols = ["plan_type", "billing_cycle", "region"]

for col_name in categorical_cols:
    # Replace nulls with 'unknown'
    ml_features = ml_features.withColumn(col_name, F.coalesce(F.col(col_name), F.lit("unknown")))
    
    # Get distinct categories
    categories = [row[0] for row in ml_features.select(col_name).distinct().collect()]
    
    # Create integer mapping
    mapping_expr = F.create_map([F.lit(x) for pair in zip(categories, range(len(categories))) for x in pair])
    
    ml_features = ml_features.withColumn(f"{col_name}_idx", mapping_expr[F.col(col_name)])

# Optional: One-hot encoding via when/otherwise
# for col_name in categorical_cols:
#     categories = [row[0] for row in ml_features.select(col_name).distinct().collect()]
#     for cat in categories:
#         ml_features = ml_features.withColumn(f"{col_name}_{cat}", F.when(F.col(col_name) == cat, 1).otherwise(0))

# -------------------------
# 8️⃣ Save ML-Ready Delta Table
# -------------------------
ml_features.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hetul_catalog_ott_02.gold.gold_churn_features_ml")

print("✅ ML-ready feature table created: hetul_catalog_ott_02.gold.gold_churn_features_ml")


✅ ML-ready feature table created: hetul_catalog_ott_02.gold.gold_churn_features_ml
